In [1]:
import math
from abc import ABC, abstractmethod
from pathlib import Path

import numpy as np
import requests

In [2]:
np.random.seed(42)

In [3]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        topo = []
        visited = set()
        stack = [(self, False)]

        while stack:
            node, expanded = stack.pop()
            if node in visited:
                continue

            if expanded:
                visited.add(node)
                topo.append(node)
            else:
                stack.append((node, True))
                for p in node.parents:
                    if p not in visited:
                        stack.append((p, False))

        self.grad = np.ones_like(self.data)
        for t in reversed(topo):
            if t.gradient_fn is not None:
                t.gradient_fn()

        for t in topo:
            t.gradient_fn = lambda: None
            t.parents = set()

    @property
    def shape(self):
        return self.data.shape

    def __add__(self, other):
        p = Tensor(self.data + other.data)

        def gradient_fn():
            self.grad += p.grad
            other.grad += p.grad

        p.gradient_fn = gradient_fn
        p.parents = {self, other}
        return p

    def __str__(self):
        return f'Tensor({self.data})'

In [4]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size
        self.load()
        self.train()

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.data = self.train_data

    def eval(self):
        self.data = self.test_data

    def all(self):
        x, y = self.data
        return Tensor(x), Tensor(y)

    def __len__(self):
        return math.ceil(len(self.data[0]) / self.batch_size)

    def __getitem__(self, index):
        s = slice(index * self.batch_size, (index + 1) * self.batch_size)
        x, y = self.data
        return Tensor(x[s]), Tensor(y[s])

In [5]:
class CharDataset(Dataset):

    def __init__(self, filename, batch_size=1, context_size=64, stride=None, split=0.9):
        self.filename = filename
        self.context_size = context_size
        self.stride = stride if stride is not None else context_size // 2
        self.split = split
        super().__init__(batch_size)

    def load(self):
        with open(self.filename, encoding="utf-8") as f:
            text = f.read()

        self.vocab = sorted(set(text))
        self.vocab_size = len(self.vocab)
        self.stoi = {ch: i for i, ch in enumerate(self.vocab)}
        self.itos = {i: ch for i, ch in enumerate(self.vocab)}
        self.tokens = self.encode(text)

        split = int(len(self.tokens) * self.split)
        self.train_data = self._pack(self.tokens[:split])
        self.test_data = self._pack(self.tokens[split:])

    def _pack(self, tokens):
        onehot = np.eye(self.vocab_size)
        x, y = [], []
        for i in range(0, len(tokens) - self.context_size - 1, self.stride):
            x.append(tokens[i: i + self.context_size])
            y.append(onehot[tokens[i + 1: i + self.context_size + 1]])
        return x, y

    def encode(self, symbols):
        return [self.stoi[s] for s in symbols]

    def decode(self, tokens):
        return "".join(self.itos[t] for t in tokens)

In [6]:
class Layer(ABC):

    def __call__(self, *args):
        return self.forward(*args)

    @abstractmethod
    def forward(self, *args):
        pass

    @property
    def parameters(self):
        return []

In [7]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = Tensor(np.random.randn(out_size, in_size) * np.sqrt(2 / in_size))
        self.bias = Tensor(np.random.rand(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            grad = p.grad.reshape(-1, p.grad.shape[-1])
            self.weight.grad += grad.T @ x.data.reshape(-1, x.shape[-1])
            self.bias.grad += np.sum(grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

In [8]:
class Composite(Layer, ABC):

    def __init__(self, layers):
        super().__init__()
        self.layers = list(layers)

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

In [9]:
class Sequential(Layer):

    def __init__(self, layers):
        self.layers = layers

    def forward(self, x: Tensor):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

In [10]:
class Embedding(Layer):

    def __init__(self, vocab_size, embedding_size, std=0.02):
        super().__init__()
        self.weight = Tensor(np.random.randn(vocab_size, embedding_size) * std)

    def forward(self, x: Tensor):
        p = Tensor(self.weight.data[x.data])

        def gradient_fn():
            np.add.at(self.weight.grad, x.data, p.grad)

        p.gradient_fn = gradient_fn
        p.parents = {self.weight}
        return p

    @property
    def parameters(self):
        return [self.weight]

In [11]:
class ReLU(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.maximum(0, x.data))

        def gradient_fn():
            x.grad += a.grad * (a.data > 0)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

In [12]:
class Softmax(Layer):

    def __init__(self, axis=-1):
        super().__init__()
        self.axis = axis

    def forward(self, x: Tensor):
        exp = np.exp(x.data - np.max(x.data, axis=self.axis, keepdims=True))
        a = Tensor(exp / np.sum(exp, axis=self.axis, keepdims=True))

        def gradient_fn():
            grad = np.sum(a.data * a.grad, axis=self.axis, keepdims=True)
            x.grad += a.data * (a.grad - grad)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

In [13]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

In [14]:
class CELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        exp = np.exp(p.data - np.max(p.data, axis=-1, keepdims=True))
        softmax = exp / np.sum(exp, axis=-1, keepdims=True)

        log = np.log(np.clip(softmax, 1e-10, 1))
        ce = Tensor(0 - np.sum(y.data * log) / len(y.data))

        def gradient_fn():
            p.grad += (softmax - y.data) / len(y.data)

        ce.gradient_fn = gradient_fn
        ce.parents = {p}
        return ce

In [15]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = list(parameters)
        self.lr = lr

    @abstractmethod
    def step(self):
        pass

    def zero_grad(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

In [16]:
class AdamOptimizer(Optimizer):

    def __init__(self, parameters, lr=0.01, betas=(0.9, 0.999), eps=1e-8):
        super().__init__(parameters, lr)
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.m: list[int | None] = [None] * len(parameters)
        self.v: list[int | None] = [None] * len(parameters)
        self.t = 0

    def step(self):
        self.t += 1
        for idx, p in enumerate(self.parameters):
            if p is not None:
                if self.m[idx] is None:
                    self.m[idx] = np.zeros_like(p.data)
                    self.v[idx] = np.zeros_like(p.data)

                self.m[idx] = self.beta1 * self.m[idx] + (1 - self.beta1) * p.grad
                self.v[idx] = self.beta2 * self.v[idx] + (1 - self.beta2) * (p.grad ** 2)
                m_hat = self.m[idx] / (1 - self.beta1 ** self.t)
                v_hat = self.v[idx] / (1 - self.beta2 ** self.t)
                self._apply_weight_decay(p)
                p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

    def states(self):
        state = {"t": self.t}
        for idx, m in enumerate(self.m):
            if m is not None:
                state[f"m_{idx}"] = m
        for idx, v in enumerate(self.v):
            if v is not None:
                state[f"v_{idx}"] = v
        return state

    def load_states(self, state):
        self.t = int(state["t"])
        for idx in range(len(self.parameters)):
            if f"m_{idx}" in state:
                self.m[idx] = np.asarray(state[f"m_{idx}"])
                self.v[idx] = np.asarray(state[f"v_{idx}"])

    def _apply_weight_decay(self, p):
        pass

    def clip_grad_norm(self, max_norm=1.0):
        sq = 0.0
        for p in self.parameters:
            sq += float(np.sum(p.grad.astype(np.float64) ** 2))

        if np.sqrt(sq) > max_norm > 0:
            scale = max_norm / (np.sqrt(sq) + 1e-6)
            for p in self.parameters:
                p.grad *= scale

In [17]:
class AdamWOptimizer(AdamOptimizer):

    def __init__(self, parameters, lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        super().__init__(parameters, lr, betas, eps)
        self.weight_decay = weight_decay

    def _apply_weight_decay(self, p):
        if p.data.ndim >= 2:
            p.data -= p.data * self.weight_decay * self.lr

In [18]:
class WarmupCosineScheduler:

    def __init__(self, max_lr, total_steps, warmup_steps, min_lr=0.0):
        self.max_lr = max_lr
        self.total_steps = max(total_steps, 1)
        self.warmup_steps = max(min(warmup_steps, self.total_steps), 0)
        self.min_lr = min_lr

    def step(self, current_step):
        if self.warmup_steps > 0 and current_step < self.warmup_steps:
            return self.max_lr * (current_step + 1) / self.warmup_steps

        if current_step >= self.total_steps:
            return self.min_lr

        progress = (current_step - self.warmup_steps) / max(self.total_steps - self.warmup_steps, 1)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return self.min_lr + (self.max_lr - self.min_lr) * cosine

In [19]:
class GPTEmbedding(Composite):

    def __init__(self, vocab_size, embedding_size):
        self.embedding = Embedding(vocab_size, embedding_size)

        super().__init__([self.embedding])

    def forward(self, x: Tensor):
        return self.embedding(x)

In [20]:
class GPTTransformer(Composite):

    def __init__(self, embedding_size):
        self.input = Linear(embedding_size, embedding_size * 4)
        self.relu = ReLU()
        self.output = Linear(embedding_size * 4, embedding_size)

        super().__init__([self.input,
                          self.relu,
                          self.output])

    def forward(self, x: Tensor):
        h = self.relu(self.input(x))
        return self.output(h)

In [21]:
class GPTOutput(Composite):

    def __init__(self, embedding_size, vocab_size):
        self.output = Linear(embedding_size, vocab_size)

        super().__init__([self.output])

    def forward(self, x: Tensor):
        return self.output(x)

In [22]:
class GPT(Composite):

    def __init__(self, vocab_size, embedding_size, blocks):
        self.embedding = GPTEmbedding(vocab_size, embedding_size)
        self.transformers = [GPTTransformer(embedding_size) for _ in range(blocks)]
        self.output = GPTOutput(embedding_size, vocab_size)

        super().__init__([self.embedding] + self.transformers + [self.output])

    def forward(self, x: Tensor, h: Tensor = None):
        x = self.embedding(x)
        for layer in self.transformers:
            x = layer(x)
        return self.output(x)

In [23]:
class GPTModel:

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    def train(self, dataset, epochs, scheduler=None):
        dataset.train()

        steps = 0
        for epoch in range(epochs):
            for i in range(len(dataset)):
                if scheduler is not None:
                    self.optimizer.lr = scheduler.step(steps)

                feature, label = dataset[i]

                self.optimizer.zero_grad()
                prediction = self.layer(feature)
                loss = self.loss_fn(prediction, label)
                loss.backward()
                self.optimizer.clip_grad_norm()
                self.optimizer.step()
                steps += 1

    def evaluate(self, dataset):
        dataset.eval()

        predictions = []
        total_loss = 0.0
        for i in range(len(dataset)):
            feature, label = dataset[i]
            prediction = self.layer(feature)
            loss = self.loss_fn(prediction, label)
            predictions.append(prediction)
            total_loss += float(loss.data)

        dataset.train()
        return predictions, total_loss / len(dataset)

    def generate(self, dataset, prompt, steps=300):
        tokens = dataset.encode(prompt)

        for _ in range(steps):
            feature = Tensor([tokens[-dataset.context_size:]])
            logits = self.layer(feature)

            last_logits = logits.data[0, -1]
            exp = np.exp(last_logits - np.max(last_logits))
            probs = exp / np.sum(exp)
            next_token = np.random.choice(len(probs), p=probs)
            tokens.append(next_token)

        return dataset.decode(tokens)

In [24]:
DATA_FILE = "../../tinyshakespeare.txt"

In [25]:
LEARNING_RATE = 0.0005

In [26]:
BATCH_SIZE = 4

In [27]:
CONTEXT_SIZE = 32

In [28]:
EMBEDDING_SIZE = 64

In [29]:
BLOCKS = 3

In [30]:
EPOCHS = 2

In [31]:
file = Path(DATA_FILE)
if not file.exists():
    file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    response = requests.get(url)
    response.raise_for_status()
    file.write_text(response.text)

In [32]:
dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
layer = GPT(dataset.vocab_size, EMBEDDING_SIZE, BLOCKS)
loss_fn = CELoss()
optimizer = AdamWOptimizer(layer.parameters, lr=LEARNING_RATE)
model = GPTModel(layer, loss_fn, optimizer)

In [33]:
scheduler = WarmupCosineScheduler(LEARNING_RATE, EPOCHS * len(dataset), 100, LEARNING_RATE / 10)
model.train(dataset, EPOCHS, scheduler)

In [34]:
prediction, loss = model.evaluate(dataset)

In [35]:
print(f'prediction: {len(prediction)} steps, each {prediction[0].shape}')
print(f'loss: {loss}')

prediction: 1743 steps, each (4, 32, 65)
loss: 8.73486275724388


In [36]:
print(model.generate(dataset, prompt="ROMEO:", steps=300))

ROMEO: landioube d heducomeaite n sano n hotindlious n,

Hoo arr, w,
DUCISo o?
I wher, ban he
CHer, gh!
STAs
As tthe aus re d may s, f wild toutiler cur ay, the me lllior thed ber l t!

BTHO:
Nod ghiknd t?
HLLUMalleesother
Filme y irealkn:
Cayop, MI'd it;

IARe we milo hot iler d y fopthippe, t y,
Tronta 
